# Agent 365 - CSV Lander (manual upload only - fallback)

> **Fallback path — prefer [`Copilot_Agent365_Registry_Ingester.ipynb`](./Copilot_Agent365_Registry_Ingester.ipynb)** for
> production. The Ingester pulls Agent 365 live from Microsoft Graph (app-only, scheduled) and needs
> no CSV upload step. Use **this** Lander only when you can't grant the app-registration permissions
> the Ingester requires, or for one-off / evaluation runs from a static export.

Lands the **Agents 365** registry export into the Lakehouse Delta table `dbo.agents_365`, which the
Fabric dashboard reads via `FabricTable("agents_365")`. The two notebooks are **alternatives** — they
target the same Delta table, so running both in the same pipeline would just clobber each other.

Keeping Agent 365 in the Lakehouse (rather than a SharePoint URL in the report) keeps the Fabric
model **100% Lakehouse-sourced** — no gateway, no privacy-firewall, Direct Lake eligible.

**How to use:** drop the Agents 365 CSV at `Files/agent365/agents.csv` (or point `SOURCE_CSV` at a
OneLake shortcut), attach this notebook to the `<your-lakehouse>` Lakehouse, and Run all.
If the file is absent the dashboard's Agents 365 table simply loads empty (it is an optional source).

In [ ]:
# === CONFIG ===
SOURCE_CSV   = 'Files/agent365/agents.csv'   # drop the Agents 365 (MAC) export here, or a OneLake shortcut path
OUTPUT_TABLE = 'dbo.agents_365'              # Delta table read by the dashboard's Agents 365 table
WRITE_MODE   = 'overwrite'                   # 'overwrite' for full snapshots; 'append' for incremental

In [ ]:
# === LAND CSV -> Delta =========================================================
# Resilient to Agent 365 schema drift.
#
#   * every source column is kept, exactly as exported
#   * canonical names are added as COPIES, never renames, so no source column is
#     consumed (renaming 'Publisher' -> 'Agent creator' used to blank Publisher)
#   * header matching ignores case, spaces and punctuation, so 'Creator ID',
#     'Creator Id' and 'creator_id' all resolve
#   * ambiguous case-insensitive duplicates are rejected unless every duplicate
#     column is value-equivalent, which avoids Spark's case-insensitive ambiguity
#   * the match report below states what resolved, what fell back and what is
#     missing, so a rename is visible at ingestion instead of weeks later
import re
import notebookutils
from pyspark.sql import functions as F


def _safe_ls(path):
    try:
        return list(notebookutils.fs.ls(path))
    except Exception as exc:
        lowered = str(exc).lower()
        if 'not found' in lowered or 'no such file' in lowered or 'does not exist' in lowered:
            return []
        raise


def _exists(path):
    folder = '/'.join(path.split('/')[:-1])
    name = path.split('/')[-1]
    return any(item.name == name for item in _safe_ls(folder))


def _norm(value):
    return re.sub(r'[^a-z0-9]', '', str(value).lower())


def _build_alias_plan():
    canonical_cols = [
        'Agent name', 'Supported in', 'Date created', 'Agent creator', 'Publisher',
        'Agent type (A365)', 'Version', 'Availability', 'Agent creator ID',
        'Agent description', 'Created in', 'Last updated', 'Custom actions',
        'Title ID', 'Sensitivity',
        'Can read OneDrive and Sharepoint items', 'OneDrive and Sharepoint items',
        'Can read OneDrive files', 'OneDrive files', 'OneDrive sites',
        'Can read Sharepoint sites and files', 'Sharepoint files', 'Sharepoint sites',
        'Can extend to Graph connector', 'Graph connector details',
        'Can generate images using user prompt', 'Can use code interpreter',
        'Contains uploaded files', 'Uploaded files', 'Status',
        'Active Users', 'Total sessions', 'Exception rate', 'Last Activity Date',
        'Deployment', 'Run Time', 'Risks',
    ]
    alias_plan = {
        'Agent name': ['Agent name', 'Name', 'Agent', 'Display name', 'DisplayName'],
        'Supported in': ['Supported in', 'Supported clients', 'Supported hosts', 'Channel', 'Channels', 'Supported platforms'],
        'Date created': ['Date created', 'Created date', 'Created on', 'Created', 'CreatedDateTime'],
        'Agent creator': ['Agent creator', 'Owner', 'Created by', 'Developer Name', 'Publisher', 'Publisher name'],
        'Publisher': ['Publisher', 'Publisher name', 'Developer', 'Developer Name', 'Company'],
        'Agent type (A365)': ['Agent type (A365)', 'Agent type', 'Publisher Type', 'Type', 'Package type'],
        'Version': ['Version', 'Agent version', 'Package version'],
        'Availability': ['Availability', 'Available to', 'Availability status'],
        'Agent creator ID': ['Agent creator ID', 'Creator Id', 'Creator ID', 'Owner Id', 'Owner ID', 'Created by id', 'Created by ID'],
        'Agent description': ['Agent description', 'Description', 'Short description', 'Summary', 'Long description'],
        'Created in': ['Created in', 'Platform', 'Source', 'Created via'],
        'Last updated': ['Last updated', 'Last Modified', 'Last modified', 'Modified date', 'LastModifiedDateTime'],
        'Custom actions': ['Custom actions', 'Custom action', 'Commands', 'Actions', 'Action titles'],
        'Title ID': ['Title ID', 'Title Id', 'TitleId', 'title_id', 'Package ID', 'Package Id', 'PackageID'],
        'Sensitivity': ['Sensitivity', 'Sensitivity label', 'SensitivityLabel'],
        'Can read OneDrive and Sharepoint items': [
            'Can read OneDrive and Sharepoint items',
            'Can read OneDrive and SharePoint items',
            'Read OneDrive and SharePoint items',
        ],
        'OneDrive and Sharepoint items': [
            'OneDrive and Sharepoint items',
            'OneDrive and SharePoint items',
            'SharePoint items',
        ],
        'Can read OneDrive files': ['Can read OneDrive files', 'Read OneDrive files'],
        'OneDrive files': ['OneDrive files', 'OneDrive file list'],
        'OneDrive sites': ['OneDrive sites', 'OneDrive site list'],
        'Can read Sharepoint sites and files': [
            'Can read Sharepoint sites and files',
            'Can read SharePoint sites and files',
            'Read SharePoint sites and files',
        ],
        'Sharepoint files': ['Sharepoint files', 'SharePoint files'],
        'Sharepoint sites': ['Sharepoint sites', 'SharePoint sites'],
        'Can extend to Graph connector': ['Can extend to Graph connector', 'Can extend to Graph connectors', 'Graph connector access'],
        'Graph connector details': ['Graph connector details', 'Graph connector detail', 'Graph connectors', 'Connector details'],
        'Can generate images using user prompt': ['Can generate images using user prompt', 'Generate images', 'Image generation'],
        'Can use code interpreter': ['Can use code interpreter', 'Use code interpreter', 'Code interpreter'],
        'Contains uploaded files': ['Contains uploaded files', 'Uploaded files present', 'Has uploaded files'],
        'Uploaded files': ['Uploaded files', 'Uploaded file list'],
        'Status': ['Status', 'Deployment status'],
        'Active Users': ['Active Users', 'Active users', 'Users', 'Monthly active users'],
        'Total sessions': ['Total sessions', 'Sessions', 'Session count'],
        'Exception rate': ['Exception rate', 'Error rate'],
        'Last Activity Date': ['Last Activity Date', 'Last used', 'Last activity', 'Last seen', 'Last used date'],
        'Deployment': ['Deployment', 'Deployed to', 'Published to'],
        'Run Time': ['Run Time', 'Runtime', 'Run time', 'Total runtime', 'Total Run Time'],
        'Risks': ['Risks', 'Risk', 'Risk summary'],
    }
    return canonical_cols, alias_plan


def _build_header_groups(columns):
    groups = {}
    for column in columns:
        groups.setdefault(_norm(column), []).append(column)
    return groups


def _resolve_alias_targets(columns, alias_plan, canonical_cols):
    groups = _build_header_groups(columns)
    clashes = {key: cols for key, cols in groups.items() if len(cols) > 1}
    if clashes:
        readable = ', '.join(f"{cols}" for cols in clashes.values())
        raise ValueError(f'Ambiguous source headers after normalization: {readable}')
    lookup = {_norm(column): column for column in columns}
    resolved = {}
    for target in canonical_cols:
        for candidate in alias_plan[target]:
            hit = lookup.get(_norm(candidate))
            if hit is not None:
                resolved[target] = hit
                break
    return resolved


def _columns_equivalent(df, columns):
    # Address source columns positionally: Spark resolves names case-insensitively.
    positional = df.toDF(*[f'_vl_source_{i}' for i in range(len(df.columns))])
    baseline = f'_vl_source_{df.columns.index(columns[0])}'
    for other in columns[1:]:
        candidate = f'_vl_source_{df.columns.index(other)}'
        mismatch = positional.filter(
            ~F.col(baseline).cast('string').eqNullSafe(F.col(candidate).cast('string'))
        ).limit(1).count()
        if mismatch:
            return False
    return True


def _preferred_duplicate(columns, canonical_cols):
    canonical_lookup = {_norm(column): column for column in canonical_cols}
    for column in columns:
        if canonical_lookup.get(_norm(column)) == column:
            return column
    return columns[0]


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


if not _exists(SOURCE_CSV):
    print(f"Agent 365 export not found at {SOURCE_CSV} - nothing landed. "
          f"Preserving any existing {OUTPUT_TABLE} snapshot.")
else:
    df = (spark.read
          .option('header', True)
          .option('multiLine', True)
          .option('escape', '"')
          .option('encoding', 'UTF-8')
          .csv(SOURCE_CSV))

    for column in df.columns:
        if column != column.strip():
            df = df.withColumnRenamed(column, column.strip())

    canonical_cols, alias_plan = _build_alias_plan()
    header_groups = _build_header_groups(df.columns)
    duplicate_resolved = []
    duplicate_dropped = set()
    for columns in header_groups.values():
        if len(columns) == 1:
            continue
        if not _columns_equivalent(df, columns):
            raise ValueError(f'Ambiguous source headers after normalization: {columns}')
        keep = _preferred_duplicate(columns, canonical_cols)
        for column in columns:
            if column != keep:
                duplicate_dropped.add(column)
        duplicate_resolved.append((keep, [column for column in columns if column != keep]))

    source_columns = [column for column in df.columns if column not in duplicate_dropped]
    resolved_map = _resolve_alias_targets(source_columns, alias_plan, canonical_cols)

    resolved_exact, fallback, missing = [], [], []
    # Build one projection from immutable source slots. Incremental withColumn calls
    # can replace a case-only alias and accidentally duplicate it in the extras list.
    positions = {name: f'_vl_source_{i}' for i, name in enumerate(df.columns)}
    positional = df.toDF(*[f'_vl_source_{i}' for i in range(len(df.columns))])
    expressions = []
    added_null = []
    for target in canonical_cols:
        source = resolved_map.get(target)
        if source is None:
            missing.append(target)
            added_null.append(target)
            expressions.append(F.lit(None).cast('string').alias(target))
        else:
            expressions.append(F.col(positions[source]).alias(target))
            if source == target:
                resolved_exact.append(target)
            else:
                fallback.append((target, source))
    canonical_names = {name.casefold() for name in canonical_cols}
    extras = [name for name in source_columns if name.casefold() not in canonical_names]
    expressions.extend(F.col(positions[name]).alias(name) for name in extras)
    df = positional.select(*expressions)

    row_count = df.count()
    if row_count == 0 and _table_exists(OUTPUT_TABLE):
        raise ValueError(f'Agent 365 CSV parsed 0 rows; refusing to replace existing {OUTPUT_TABLE}.')
    if row_count == 0:
        raise ValueError('Agent 365 CSV parsed 0 rows; refusing to write an empty snapshot.')

    as_of = df.agg(F.max(F.to_timestamp(F.col('`Last updated`')))).collect()[0][0]
    df = df.withColumn('Snapshot As Of', F.lit(as_of).cast('timestamp'))

    print('Agent 365 schema match')
    print(f'  exact       : {len(resolved_exact)}')
    if fallback:
        print(f'  via alias   : {len(fallback)}')
        for target, source in fallback:
            print(f'                {target!r} <- {source!r}')
    if duplicate_resolved:
        print(f'  deduped hdr : {len(duplicate_resolved)}')
        for keep, dropped in duplicate_resolved:
            print(f'                {keep!r} kept; dropped equivalent {dropped!r}')
    if missing:
        print(f'  NOT FOUND   : {len(missing)} (loaded as blank)')
        for target in missing:
            print(f'                {target}')
    if extras:
        print(f'  extra cols  : {len(extras)} kept as-is')
    print(f'  snapshot as of: {as_of}')
    print(f'  rows: {row_count:,} | columns: {len(df.columns)}')

    (df.write.mode(WRITE_MODE)
       .option('overwriteSchema', 'true')
       .option('delta.columnMapping.mode', 'name')
       .option('delta.minReaderVersion', '2')
       .option('delta.minWriterVersion', '5')
       .format('delta').saveAsTable(OUTPUT_TABLE))
